In [41]:
import numpy as np
import random
from collections import deque, defaultdict

# ─── 地圖與參數 ─────────────────────────────────────────
WIDTH, HEIGHT = 21, 11
NUM_POS        = WIDTH * HEIGHT
treasure_list  = [(0, 6), (3, 16), (8, 2), (10, 2), (10, 17)]
walls_coords   = set([
    (0,4),(0,5),(0,7),(0,9),(1,1),(1,2),(1,4),(1,9),(1,10),(1,14),(1,18),
    (2,1),(2,3),(2,5),(2,7),(2,8),(2,9),(2,11),(2,13),(2,15),(2,16),(2,17),(2,19),
    (3,2),(3,8),(3,11),(3,17),(4,1),(4,4),(4,6),(4,10),(4,13),(4,16),(4,17),(4,18),(4,20),
    (5,4),(5,5),(5,6),(5,8),(5,9),(5,14),(5,15),(6,1),(6,2),(6,3),(6,6),(6,8),(6,10),(6,15),(6,16),(6,17),(6,19),
    (7,4),(7,6),(7,8),(7,10),(7,11),(7,17),(7,19),(8,1),(8,4),(8,8),(8,10),(8,13),(8,15),(8,18),(8,19),
    (9,1),(9,2),(9,4),(9,6),(9,7),(9,17),(10,1),(10,4),(10,16),(10,19)
])
treasures = set(treasure_list)
START, GOAL = (0,0), (20,10)

# Q-learning 超參數
MAX_EPISODES = 1000
MAX_STEPS    = 1000
EPSILON, EPS_DECAY, EPS_MIN = 0.9, 0.995, 0.01
ALPHA, GAMMA = 0.9, 0.995

# PER 參數
BUFFER_CAPACITY = 5000
REPLAY_BATCH    = 32
PER_EPSILON     = 1e-5
PER_UPDATE_FREQ = 50

transition_buffer = deque(maxlen=BUFFER_CAPACITY)
priority_buffer   = deque(maxlen=BUFFER_CAPACITY)

# 提前停機
patience, no_improve = 500, 0

ACTIONS = ['up','down','left','right']
NUM_ACTIONS = len(ACTIONS)
NUM_MASK    = 1 << len(treasure_list)
NUM_STATES  = NUM_POS * NUM_MASK

# 雙 Q 表
Q1 = np.zeros((NUM_STATES, NUM_ACTIONS))
Q2 = np.zeros((NUM_STATES, NUM_ACTIONS))

# 工具函數
def to_index(pos):
    return pos[1] * WIDTH + pos[0]

def to_pos(idx):
    return (idx % WIDTH, idx // WIDTH)

def encode_state(pos, mask):
    return to_index(pos) * NUM_MASK + mask

def compute_target(pos, act):
    x, y = pos
    if act == 'up':
        y = max(0, y-1)
    elif act == 'down':
        y = min(HEIGHT-1, y+1)
    elif act == 'left':
        x = max(0, x-1)
    else:
        x = min(WIDTH-1, x+1)
    return (x, y)

# ─── 新增：BFS 計算從任意格子到子目標的最短距離 ───────────────
def bfs_distances(target):
    dist = { (x,y): float('inf') for x in range(WIDTH) for y in range(HEIGHT) }
    dq = deque([target])
    dist[target] = 0
    while dq:
        x,y = dq.popleft()
        for dx,dy in [(0,1),(0,-1),(1,0),(-1,0)]:
            nx,ny = x+dx, y+dy
            if 0 <= nx < WIDTH and 0 <= ny < HEIGHT and (nx,ny) not in walls_coords:
                if dist[(nx,ny)] > dist[(x,y)] + 1:
                    dist[(nx,ny)] = dist[(x,y)] + 1
                    dq.append((nx,ny))
    return dist

# 緩存每個 mask 對應的距離表
bfs_cache = {}

def compute_potential(pos, mask):
    # 決定下一個目標：最近未拿寶 or GOAL
    rem = [t for i,t in enumerate(treasure_list) if not (mask & (1<<i))]
    target = min(rem, key=lambda t: abs(t[0]-pos[0]) + abs(t[1]-pos[1])) if rem else GOAL
    # 如果還沒計算這個 mask 對應的距離表，就 bfs 一次
    if mask not in bfs_cache:
        bfs_cache[mask] = bfs_distances(target)
    d = bfs_cache[mask].get(pos, float('inf'))
    return -d      # φ = –distance

# ─── 主訓練迴圈 ────────────────────────────────────────────────
best_steps, best_score, best_path = MAX_STEPS+1, -1, None
best_treasure_score, best_treasure_path = -1, None
epsilon = EPSILON

for ep in range(1, MAX_EPISODES+1):
    pos, mask = START, 0
    state     = encode_state(pos, mask)
    path      = [state]
    score     = 0
    improved  = False
    visit_count = defaultdict(int)
    visit_count[state] = 1

    # 初始化 φ(s)
    phi_s = compute_potential(pos, mask)

    for step in range(1, MAX_STEPS+1):
        # ε-貪婪 + 障礙回避
        while True:
            if random.random() < epsilon:
                a = random.randrange(NUM_ACTIONS)
            else:
                a = np.argmax(Q1[state] + Q2[state])
            new_pos = compute_target(pos, ACTIONS[a])
            if new_pos in walls_coords or new_pos == pos:
                continue
            break

        # 更新 new_mask
        new_mask = mask
        if new_pos in treasures:
            i = treasure_list.index(new_pos)
            if not (mask & (1<<i)):
                new_mask |= (1<<i)

        # ————— 原有基本獎勵 + 獎勵塑形 + 迴圈懲罰 —————
        reward = -1
        if new_pos in treasures:
            i = treasure_list.index(new_pos)
            if not (mask & (1<<i)):
                reward = +10; score += 1
        elif new_pos == GOAL:
            reward = +58 if new_mask == (NUM_MASK-1) else -50

        # 額外迴圈訪問懲罰
        new_state = encode_state(new_pos, new_mask)
        visit_count[new_state] += 1
        if visit_count[new_state] >= 2:
            reward -= 500

        # ————— 潛在函數獎勵塑形 —————
        phi_s2 = compute_potential(new_pos, new_mask)
        reward += GAMMA * phi_s2 - phi_s
        phi_s = phi_s2

        # Double Q 更新
        if random.random() < 0.5:
            an = np.argmax(Q1[new_state])
            td = reward + GAMMA * Q2[new_state, an] - Q1[state, a]
            Q1[state, a] += ALPHA * td
        else:
            an = np.argmax(Q2[new_state])
            td = reward + GAMMA * Q1[new_state, an] - Q2[state, a]
            Q2[state, a] += ALPHA * td

        # PER 存儲與更新（放在原來的那段代碼裡）
        transition_buffer.append((state, a, reward, new_state))
        priority_buffer.append(abs(td) + PER_EPSILON)

        if step % PER_UPDATE_FREQ == 0 and len(transition_buffer) >= REPLAY_BATCH:
            # 構造概率向量
            p = np.array(priority_buffer, dtype=float)
            # 把 NaN 轉成一個很小的正數
            p = np.nan_to_num(p, nan=PER_EPSILON, posinf=PER_EPSILON, neginf=PER_EPSILON)
            total = p.sum()
            # 如果出現了 sum == 0，也退化到均勻分佈
            if total <= 0:
                p = np.ones_like(p)
                total = p.sum()
            p /= total

            # 採樣
            idxs = np.random.choice(len(p), REPLAY_BATCH, p=p)
            for idx in idxs:
                s, a0, r0, sn = transition_buffer[idx]
                if random.random() < 0.5:
                    an0 = np.argmax(Q1[sn])
                    Q1[s, a0] += ALPHA * (r0 + GAMMA * Q2[sn, an0] - Q1[s, a0])
                else:
                    an0 = np.argmax(Q2[sn])
                    Q2[s, a0] += ALPHA * (r0 + GAMMA * Q1[sn, an0] - Q2[s, a0])

        pos, mask, state = new_pos, new_mask, new_state
        path.append(state)

        # 成功記錄
        if new_pos == GOAL and new_mask == (NUM_MASK-1):
            improved = True
            if step < best_steps:
                best_steps, best_score, best_path = step, score, path.copy()
            break

    # 更新“最佳寶藏數”路徑
    if score > best_treasure_score:
        best_treasure_score, best_treasure_path = score, path.copy()

    # ε 衰減 & 早停
    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)
    no_improve = 0 if improved else no_improve + 1
    if epsilon <= EPS_MIN and no_improve >= patience:
        break

# ─── 輸出最終結果 ───────────────────────────────────────────
if best_path is not None:
    use_path, label, final_score = best_path, "（成功）", best_score
else:
    use_path, label, final_score = best_treasure_path, "（未成功，以寶藏數最多輸出）", best_treasure_score

# 重播路徑，收集寶藏並記錄最終位置
collected = []
pos, mask = START, 0
for s in use_path:
    idx = s // NUM_MASK
    p   = to_pos(idx)
    if p in treasure_list and p not in collected:
        collected.append(p)
    pos = p

print(f"=== 最終結果 {label} ===")
print(f"步數：{len(use_path)-1}")
print(f"寶藏：{final_score}/{len(treasure_list)} -> {collected}")
print(f"最終停留：{pos} (應為 {GOAL})")

C:\Users\Yves\AppData\Local\Temp\ipykernel_20284\2803492060.py:160: RuntimeWarning: invalid value encountered in scalar subtract
  td = reward + GAMMA * Q1[new_state, an] - Q2[state, a]


=== 最終結果 （未成功，以寶藏數最多輸出） ===
步數：1000
寶藏：3/5 -> [(8, 2), (0, 6), (10, 2)]
最終停留：(12, 4) (應為 (20, 10))
